In [17]:
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.backends import default_backend
import snowflake.connector
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

key_path = "/home/jovyan/work/snowflake_key.p8"

# 2. read the access key
with open(key_path, "rb") as key:
    p_key = serialization.load_pem_private_key(
        key.read(),
        password=None, 
        backend=default_backend()
    )

pkb = p_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()
)

# 3. connect
conn = snowflake.connector.connect(
    user='AIRFLOW_USER',
    account='BNNZHHH-KJ98615',
    private_key=pkb,
    warehouse='EE_COMPUTE_WH',
    database='EMPLOYEE_DATA_DB',
    schema='RAW'
)

In [32]:
from sqlalchemy import create_engine
from snowflake.sqlalchemy import URL

engine = create_engine(URL(
    account='BNNZHHH-KJ98615',
    user='AIRFLOW_USER',
    database='EMPLOYEE_DATA_DB',
    schema='RAW',
    warehouse='EE_COMPUTE_WH',
    role='ACCOUNTADMIN',
), connect_args={'private_key': pkb})


In [38]:
def query(sql):
    with engine.connect() as conn:
        return pd.read_sql(sql, conn)

# Just SELECT * — columns are already properly named
df_employee    = query("SELECT * FROM RAW_EMPLOYEE_INFO")
df_engagement  = query("SELECT * FROM RAW_ENGAGEMENT")
df_recruitment = query("SELECT * FROM RAW_RECRUITMENT")
df_training    = query("SELECT * FROM RAW_TRAINING")

# Verify
print(df_engagement.shape)
df_engagement.head()

(3000, 5)


,Employee ID,Survey Date,Engagement Score,Satisfaction Score,Work-Life Balance Score
0,1001,10-10-2022,2,5,5
1,1002,03-08-2023,4,5,3
2,1003,03-01-2023,2,5,2
3,1004,30-07-2023,3,5,3
4,1005,19-06-2023,2,4,5


In [44]:
# shape and dtypes
for name, df in [
    ('EMPLOYEE', df_employee),
    ('ENGAGEMENT', df_engagement),
    ('RECRUITMENT', df_recruitment),
    ('TRAINING', df_training)
]:
    print(f"\n{'='*50}")
    print(f"{name} — shape: {df.shape}")
    print(f"{'='*50}")
    print(df.dtypes)



EMPLOYEE — shape: (3000, 26)
EmpID                          int64
FirstName                     object
LastName                      object
StartDate                     object
ExitDate                      object
Title                         object
Supervisor                    object
ADEmail                       object
BusinessUnit                  object
EmployeeStatus                object
EmployeeType                  object
PayZone                       object
EmployeeClassificationType    object
TerminationType               object
TerminationDescription        object
DepartmentType                object
Division                      object
dob                           object
State                         object
JobFunctionDescription        object
GenderCode                    object
LocationCode                   int64
RaceDesc                      object
MaritalDesc                   object
Performance Score             object
Current Employee Rating        int64
dtype: o

In [45]:
# Null check
for name, df in [
    ('EMPLOYEE', df_employee),
    ('ENGAGEMENT', df_engagement),
    ('RECRUITMENT', df_recruitment),
    ('TRAINING', df_training)
]:
    print(f"\n{'='*50}")
    print(f"{name} — null counts")
    print(f"{'='*50}")
    nulls = df.isnull().sum()
    nulls_pct = (df.isnull().sum() / len(df) * 100).round(2)
    print(pd.DataFrame({'null_count': nulls, 'null_%': nulls_pct})
            .query('null_count > 0')
            .sort_values('null_%', ascending=False))


EMPLOYEE — null counts
                        null_count  null_%
ExitDate                      1467    48.9
TerminationDescription        1467    48.9

ENGAGEMENT — null counts
Empty DataFrame
Columns: [null_count, null_%]
Index: []

RECRUITMENT — null counts
Empty DataFrame
Columns: [null_count, null_%]
Index: []

TRAINING — null counts
Empty DataFrame
Columns: [null_count, null_%]
Index: []


In [49]:
# Sample
print("\n── EMPLOYEE ──")
display(df_employee.head(3))

print("\n── ENGAGEMENT ──")
display(df_engagement.head(3))

print("\n── RECRUITMENT ──")
display(df_recruitment.head(3))

print("\n── TRAINING ──")
display(df_training.head(3))


── EMPLOYEE ──


,EmpID,FirstName,LastName,StartDate,ExitDate,Title,Supervisor,ADEmail,BusinessUnit,EmployeeStatus,...,Division,dob,State,JobFunctionDescription,GenderCode,LocationCode,RaceDesc,MaritalDesc,Performance Score,Current Employee Rating
0,3427,Uriah,Bridges,0019-09-20,None,Production Technician I,Peter Oneill,uriah.bridges@bilearner.com,CCDR,Active,...,Finance & Accounting,07-10-1969,MA,Accounting,Female,34904,White,Widowed,Fully Meets,4
1,3428,Paula,Small,0023-02-11,None,Production Technician I,Renee Mccormick,paula.small@bilearner.com,EW,Active,...,Aerial,30-08-1965,MA,Labor,Male,6593,Hispanic,Widowed,Fully Meets,3
2,3429,Edward,Buck,0018-12-10,None,Area Sales Manager,Crystal Walker,edward.buck@bilearner.com,PL,Active,...,General - Sga,06-10-1991,MA,Assistant,Male,2330,Hispanic,Widowed,Fully Meets,4



── ENGAGEMENT ──


,Employee ID,Survey Date,Engagement Score,Satisfaction Score,Work-Life Balance Score
0,1001,10-10-2022,2,5,5
1,1002,03-08-2023,4,5,3
2,1003,03-01-2023,2,5,2



── RECRUITMENT ──


,Applicant ID,Application Date,First Name,Last Name,Gender,Date of Birth,Phone Number,Email,Address,City,State,Zip Code,Country,Education Level,Years of Experience,Desired Salary,Job Title,Status
0,1001,0023-06-03,Scott,Sheppard,Male,31-08-1992,421-429-7655x39421,perezjanet@example.org,597 Smith Point,Hollandfort,NV,57588,Micronesia,High School,8,60103.21,Chief Technology Officer,Interviewing
1,1002,0023-05-15,Stanley,Lewis,Male,29-04-1965,+1-451-574-5308x1681,grossmark@example.com,8116 Stuart Loop,Port Margaretfurt,TN,14726,Greenland,Bachelor's Degree,17,64575.84,"Designer, furniture",Rejected
2,1003,0023-08-04,Javier,Li,Female,10-03-1973,(858)901-5499,katiemaldonado@example.com,5940 Barr Villages Suite 075,Dianaland,TX,4699,China,PhD,20,39422.71,"Sound technician, broadcasting/film/video",Rejected



── TRAINING ──


,Employee ID,Training Date,Training Program Name,Training Type,Training Outcome,Location,Trainer,Training Duration(Days),Training Cost
0,1001,0022-09-21,Customer Service,Internal,Failed,Port Greg,Amanda Daniels,4,510.83
1,1002,0023-07-19,Leadership Development,Internal,Failed,Brandonview,Brittany Chambers,2,582.37
2,1003,0023-02-24,Technical Skills,Internal,Incomplete,Port Briannahaven,Mark Roberson,4,777.06


In [50]:
# Basic stats on numberic columbs
for name, df in [
    ('EMPLOYEE',    df_employee),
    ('ENGAGEMENT',  df_engagement),
    ('RECRUITMENT', df_recruitment),
    ('TRAINING',    df_training)
]:
    print(f"\n{'='*50}")
    print(f"{name} — numeric stats")
    print(f"{'='*50}")
    display(df.describe())


EMPLOYEE — numeric stats


,EmpID,LocationCode,Current Employee Rating
count,3000.000000,3000.000000,3000.000000
mean,2500.500000,44997.180667,2.969000
std,866.169729,29987.331783,1.015078
min,1001.000000,1013.000000,1.000000
25%,1750.750000,17546.000000,2.000000
50%,2500.500000,44150.500000,3.000000
75%,3250.250000,71481.250000,3.000000
max,4000.000000,98052.000000,5.000000



ENGAGEMENT — numeric stats


,Employee ID,Engagement Score,Satisfaction Score,Work-Life Balance Score
count,3000.000000,3000.000000,3000.000000,3000.000000
mean,2500.500000,2.939667,3.022000,2.989000
std,866.169729,1.433426,1.408845,1.409329
min,1001.000000,1.000000,1.000000,1.000000
25%,1750.750000,2.000000,2.000000,2.000000
50%,2500.500000,3.000000,3.000000,3.000000
75%,3250.250000,4.000000,4.000000,4.000000
max,4000.000000,5.000000,5.000000,5.000000



RECRUITMENT — numeric stats


,Applicant ID,Zip Code,Years of Experience,Desired Salary
count,3000.000000,3000.000000,3000.000000,3000.000000
mean,2500.500000,51095.088000,9.964667,65079.057560
std,866.169729,28709.871983,6.039998,20163.675071
min,1001.000000,541.000000,0.000000,30047.220000
25%,1750.750000,26856.000000,5.000000,47307.807500
50%,2500.500000,51418.000000,10.000000,64934.865000
75%,3250.250000,76241.250000,15.000000,82585.595000
max,4000.000000,99897.000000,20.000000,99992.660000



TRAINING — numeric stats


,Employee ID,Training Duration(Days),Training Cost
count,3000.000000,3000.000000,3000.000000
mean,2500.500000,2.975667,558.628697
std,866.169729,1.417890,263.217698
min,1001.000000,1.000000,100.040000
25%,1750.750000,2.000000,327.587500
50%,2500.500000,3.000000,572.125000
75%,3250.250000,4.000000,786.987500
max,4000.000000,5.000000,999.970000


In [51]:
# Check duplicates
for name, df, key in [
    ('EMPLOYEE',   df_employee,   'EmpID'),
    ('ENGAGEMENT', df_engagement, 'Employee ID'),
    ('RECRUITMENT',df_recruitment,'Applicant ID'),
    ('TRAINING',   df_training,   'Employee ID')
]:
    total_rows = len(df)
    unique_keys = df[key].nunique()
    dupes = total_rows - unique_keys
    print(f"{name} — total: {total_rows}, unique {key}: {unique_keys}, duplicates: {dupes}")


EMPLOYEE — total: 3000, unique EmpID: 3000, duplicates: 0
ENGAGEMENT — total: 3000, unique Employee ID: 3000, duplicates: 0
RECRUITMENT — total: 3000, unique Applicant ID: 3000, duplicates: 0
TRAINING — total: 3000, unique Employee ID: 3000, duplicates: 0


In [52]:
for name, df in [
    ('EMPLOYEE',    df_employee),
    ('ENGAGEMENT',  df_engagement),
    ('RECRUITMENT', df_recruitment),
    ('TRAINING',    df_training)
]:
    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    for col in df.columns.tolist():
        print(f"  {col} — {df[col].dtype} — nulls: {df[col].isnull().sum()}")


EMPLOYEE
  EmpID — int64 — nulls: 0
  FirstName — object — nulls: 0
  LastName — object — nulls: 0
  StartDate — object — nulls: 0
  ExitDate — object — nulls: 1467
  Title — object — nulls: 0
  Supervisor — object — nulls: 0
  ADEmail — object — nulls: 0
  BusinessUnit — object — nulls: 0
  EmployeeStatus — object — nulls: 0
  EmployeeType — object — nulls: 0
  PayZone — object — nulls: 0
  EmployeeClassificationType — object — nulls: 0
  TerminationType — object — nulls: 0
  TerminationDescription — object — nulls: 1467
  DepartmentType — object — nulls: 0
  Division — object — nulls: 0
  dob — object — nulls: 0
  State — object — nulls: 0
  JobFunctionDescription — object — nulls: 0
  GenderCode — object — nulls: 0
  LocationCode — int64 — nulls: 0
  RaceDesc — object — nulls: 0
  MaritalDesc — object — nulls: 0
  Performance Score — object — nulls: 0
  Current Employee Rating — int64 — nulls: 0

ENGAGEMENT
  Employee ID — int64 — nulls: 0
  Survey Date — object — nulls: 0
  Engage

In [53]:
print(df_employee['GenderCode'].unique())

['Female' 'Male']


In [54]:
print(df_engagement.columns.tolist())

['Employee ID', 'Survey Date', 'Engagement Score', 'Satisfaction Score', 'Work-Life Balance Score']


In [55]:
print(df_engagement['Survey Date'].head(10))
print(df_engagement['Survey Date'].unique()[:10])


0    10-10-2022
1    03-08-2023
2    03-01-2023
3    30-07-2023
4    19-06-2023
5    03-05-2023
6    18-07-2023
7    21-06-2023
8    06-06-2023
9    15-09-2022
Name: Survey Date, dtype: object
['10-10-2022' '03-08-2023' '03-01-2023' '30-07-2023' '19-06-2023'
 '03-05-2023' '18-07-2023' '21-06-2023' '06-06-2023' '15-09-2022']


In [56]:
print(df_employee['GenderCode'].value_counts())

GenderCode
Female    1682
Male      1318
Name: count, dtype: int64


In [57]:
!ls -al /home/jovyan/work/.ipynb_checkpoints/

total 4
drwxr-xr-x  3 jovyan users  96 Apr 20 22:29 .
drwxr-xr-x 30 jovyan users 960 Apr 20 23:59 ..
-rw-r--r--  1 jovyan users  72 Apr 20 20:57 EDA-checkpoint.ipynb


In [58]:
!cp /home/jovyan/work/.ipynb_checkpoints/EDA-checkpoint.ipynb /home/jovyan/work/RECOVERY_EDA.ipynb